In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, signal
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import seaborn as sns
from tqdm.notebook import tqdm
import torch
import re
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn')
sns.set_palette('husl')


In [ ]:
# Directory paths
csi_directory = '../data_capture/csi/top_csi_filtered'
audio_directory = '../data_capture/audio_matrix'

# Helper to extract timestamp from filename
def extract_timestamp(filename):
    # For audio: 2024-05-07_20-11-50.303.csv
    # For csi: top8_filtered_csi_data_2024-05-07_20-11-50.303.csv
    match = re.search(r'(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}\.\d+)', filename)
    return match.group(1) if match else None

# Build lookup for CSI files by timestamp
csi_files = [f for f in os.listdir(csi_directory) if f.startswith('top8_filtered_csi_data_') and f.endswith('.csv')]
csi_timestamps = {extract_timestamp(f): f for f in csi_files}

audio_files = [f for f in os.listdir(audio_directory) if f.endswith('.csv')]
audio_timestamps = {extract_timestamp(f): f for f in audio_files}

# Find common timestamps
common_timestamps = sorted(set(csi_timestamps) & set(audio_timestamps))

print(f"Found {len(common_timestamps)} matched CSI-Audio pairs")


In [ ]:
class WindowAnalyzer:
    def __init__(self, window_size=100, overlap=0.5, sampling_rate=None):
        """
        Initialize the window analyzer.
        
        Args:
            window_size (int): Number of samples in each window
            overlap (float): Overlap between consecutive windows (0 to 1)
            sampling_rate (float): Sampling rate of the data in Hz
        """
        self.window_size = window_size
        self.overlap = overlap
        self.sampling_rate = sampling_rate
        self.hop_size = int(window_size * (1 - overlap))
    
    def extract_windows(self, data):
        """Extract overlapping windows from data."""
        n_samples = len(data)
        n_windows = ((n_samples - self.window_size) // self.hop_size) + 1
        return np.array([
            data[i * self.hop_size : i * self.hop_size + self.window_size]
            for i in range(n_windows)
        ])
    
    def compute_time_features(self, window):
        """Compute time-domain features."""
        return {
            'mean': np.mean(window, axis=0),
            'std': np.std(window, axis=0),
            'max': np.max(window, axis=0),
            'min': np.min(window, axis=0),
            'median': np.median(window, axis=0),
            'skewness': stats.skew(window, axis=0),
            'kurtosis': stats.kurtosis(window, axis=0),
            'rms': np.sqrt(np.mean(np.square(window), axis=0)),
            'peak_to_peak': np.ptp(window, axis=0),
            'crest_factor': np.max(np.abs(window), axis=0) / np.sqrt(np.mean(np.square(window), axis=0))
        }
    
    def compute_freq_features(self, window):
        """Compute frequency-domain features."""
        features = {}
        if self.sampling_rate:
            for i in range(window.shape[1]):
                yf = fft(window[:, i])
                xf = fftfreq(self.window_size, 1/self.sampling_rate)
                
                pos_mask = xf > 0
                yf = np.abs(yf[pos_mask])
                xf = xf[pos_mask]
                
                features[f'dominant_freq_{i}'] = xf[np.argmax(yf)]
                features[f'spectral_centroid_{i}'] = np.sum(xf * yf) / np.sum(yf)
                features[f'spectral_bandwidth_{i}'] = np.sqrt(np.sum(((xf - features[f'spectral_centroid_{i}'])**2) * yf) / np.sum(yf))
        return features
    
    def analyze_pair(self, csi_data, audio_data):
        """Analyze a matched CSI-Audio pair."""
        # Normalize data
        csi_norm = StandardScaler().fit_transform(csi_data)
        audio_norm = StandardScaler().fit_transform(audio_data.reshape(-1, 1)).flatten()
        
        # Extract windows
        csi_windows = self.extract_windows(csi_norm)
        audio_windows = self.extract_windows(audio_norm)
        
        # Compute features
        csi_time_features = [self.compute_time_features(w) for w in csi_windows]
        csi_freq_features = [self.compute_freq_features(w) for w in csi_windows]
        
        audio_time_features = [self.compute_time_features(w.reshape(-1, 1)) for w in audio_windows]
        audio_freq_features = [self.compute_freq_features(w.reshape(-1, 1)) for w in audio_windows]
        
        return {
            'csi_windows': csi_windows,
            'audio_windows': audio_windows,
            'csi_time_features': csi_time_features,
            'csi_freq_features': csi_freq_features,
            'audio_time_features': audio_time_features,
            'audio_freq_features': audio_freq_features
        }


In [ ]:
def plot_spectrograms(windows, sampling_rate, title, n_cols=2):
    """Plot spectrograms for multiple channels."""
    n_channels = windows.shape[2] if len(windows.shape) > 2 else 1
    n_rows = (n_channels + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    fig.suptitle(title, fontsize=16)
    
    if n_channels == 1:
        axes = np.array([axes])
    
    for i in range(n_channels):
        ax = axes[i // n_cols, i % n_cols] if n_rows > 1 else axes[i]
        
        data = windows[:, :, i].flatten() if len(windows.shape) > 2 else windows.flatten()
        f, t, Sxx = signal.spectrogram(
            data,
            fs=sampling_rate,
            nperseg=256,
            noverlap=128
        )
        
        im = ax.pcolormesh(t, f, 10 * np.log10(Sxx), shading='gouraud')
        ax.set_title(f'Channel {i+1}')
        ax.set_ylabel('Frequency [Hz]')
        ax.set_xlabel('Time [sec]')
        plt.colorbar(im, ax=ax)
    
    # Hide empty subplots
    for i in range(n_channels, n_rows * n_cols):
        if n_rows > 1:
            axes[i // n_cols, i % n_cols].set_visible(False)
        elif i < len(axes):
            axes[i].set_visible(False)
    
    plt.tight_layout()
    return fig

def plot_feature_correlation(features, title):
    """Plot correlation matrix of features."""
    # Convert features to DataFrame
    feature_df = pd.DataFrame([{k: v.mean() if isinstance(v, np.ndarray) else v 
                              for k, v in f.items()} for f in features])
    
    # Compute correlation matrix
    corr = feature_df.corr()
    
    # Plot
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    return plt.gcf()

def plot_feature_distributions(features, title):
    """Plot distributions of features."""
    feature_df = pd.DataFrame([{k: v.mean() if isinstance(v, np.ndarray) else v 
                              for k, v in f.items()} for f in features])
    
    n_features = len(feature_df.columns)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    fig.suptitle(title, fontsize=16)
    
    for i, col in enumerate(feature_df.columns):
        ax = axes[i // n_cols, i % n_cols]
        sns.histplot(feature_df[col], ax=ax)
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=45)
    
    # Hide empty subplots
    for i in range(n_features, n_rows * n_cols):
        axes[i // n_cols, i % n_cols].set_visible(False)
    
    plt.tight_layout()
    return fig


In [ ]:
# Initialize analyzer with appropriate parameters
analyzer = WindowAnalyzer(
    window_size=100,  # 100 samples per window
    overlap=0.5,      # 50% overlap between windows
    sampling_rate=100 # Assuming 100Hz sampling rate
)

# Process first pair as example
ts = common_timestamps[0]
print(f"Processing pair from timestamp: {ts}")

# Load CSI data
csi_file = os.path.join(csi_directory, csi_timestamps[ts])
csi_data = pd.read_csv(csi_file)
subcarrier_cols = [col for col in csi_data.columns if col.startswith('subcarrier_')]
csi_matrix = csi_data[subcarrier_cols].values

# Load audio data
audio_file = os.path.join(audio_directory, audio_timestamps[ts])
audio_data = pd.read_csv(audio_file, header=None).values.flatten()

print(f"CSI data shape: {csi_matrix.shape}")
print(f"Audio data shape: {audio_data.shape}")

# Analyze pair
results = analyzer.analyze_pair(csi_matrix, audio_data)


In [ ]:
# Plot CSI spectrograms
print("CSI Spectrograms:")
fig = plot_spectrograms(
    results['csi_windows'],
    analyzer.sampling_rate,
    'CSI Spectrograms'
)
plt.show()

# Plot audio spectrogram
print("\nAudio Spectrogram:")
fig = plot_spectrograms(
    results['audio_windows'],
    analyzer.sampling_rate,
    'Audio Spectrogram'
)
plt.show()

# Plot CSI feature correlations
print("\nCSI Feature Correlations:")
fig = plot_feature_correlation(
    results['csi_time_features'],
    'CSI Time-Domain Feature Correlations'
)
plt.show()

# Plot CSI feature distributions
print("\nCSI Feature Distributions:")
fig = plot_feature_distributions(
    results['csi_time_features'],
    'CSI Time-Domain Feature Distributions'
)
plt.show()


In [ ]:
def compute_cross_correlation(csi_features, audio_features):
    """Compute cross-correlation between CSI and audio features."""
    # Convert features to DataFrames
    csi_df = pd.DataFrame([{k: v.mean() if isinstance(v, np.ndarray) else v 
                           for k, v in f.items()} for f in csi_features])
    audio_df = pd.DataFrame([{k: v.mean() if isinstance(v, np.ndarray) else v 
                            for k, v in f.items()} for f in audio_features])
    
    # Compute correlations
    correlations = pd.DataFrame()
    for csi_col in csi_df.columns:
        for audio_col in audio_df.columns:
            corr = np.corrcoef(csi_df[csi_col], audio_df[audio_col])[0, 1]
            correlations.loc[csi_col, audio_col] = corr
    
    return correlations

# Compute and plot cross-correlations
print("CSI-Audio Feature Cross-Correlations:")
cross_corr = compute_cross_correlation(
    results['csi_time_features'],
    results['audio_time_features']
)

plt.figure(figsize=(12, 8))
sns.heatmap(cross_corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('CSI-Audio Feature Cross-Correlations')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Print top correlations
print("\nTop 5 strongest correlations:")
corr_values = []
for i in range(cross_corr.shape[0]):
    for j in range(cross_corr.shape[1]):
        corr_values.append({
            'csi_feature': cross_corr.index[i],
            'audio_feature': cross_corr.columns[j],
            'correlation': abs(cross_corr.iloc[i, j])
        })

top_corr = pd.DataFrame(corr_values).sort_values('correlation', ascending=False).head()
print(top_corr)


In [ ]:
from sklearn.decomposition import PCA

def analyze_feature_importance(features, title):
    """Analyze feature importance using PCA."""
    # Convert features to DataFrame
    feature_df = pd.DataFrame([{k: v.mean() if isinstance(v, np.ndarray) else v 
                              for k, v in f.items()} for f in features])
    
    # Standardize the features
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(feature_df)
    
    # Apply PCA
    pca = PCA()
    pca.fit(scaled_features)
    
    # Plot explained variance ratio
    plt.figure(figsize=(10, 6))
    cumulative_variance_ratio = np.cumsum(pca.explained_variance_ratio_)
    plt.plot(range(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, 'bo-')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance Ratio')
    plt.title(f'{title}\nPCA Explained Variance Ratio')
    plt.grid(True)
    plt.show()
    
    # Feature importance based on first principal component
    importance = pd.DataFrame({
        'feature': feature_df.columns,
        'importance': np.abs(pca.components_[0])
    })
    importance = importance.sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=importance, x='importance', y='feature')
    plt.title(f'{title}\nFeature Importance (First Principal Component)')
    plt.tight_layout()
    plt.show()
    
    return importance

# Analyze CSI feature importance
print("CSI Feature Importance Analysis:")
csi_importance = analyze_feature_importance(
    results['csi_time_features'],
    'CSI Time-Domain Features'
)

print("\nTop 5 most important CSI features:")
print(csi_importance.head())

# Analyze audio feature importance
print("\nAudio Feature Importance Analysis:")
audio_importance = analyze_feature_importance(
    results['audio_time_features'],
    'Audio Time-Domain Features'
)

print("\nTop 5 most important audio features:")
print(audio_importance.head())


In [ ]:
# Create results directory
results_dir = '../data_capture/window_analysis_results'
os.makedirs(results_dir, exist_ok=True)

# Save timestamp information
timestamp_info = {
    'timestamp': ts,
    'csi_file': csi_timestamps[ts],
    'audio_file': audio_timestamps[ts]
}
pd.DataFrame([timestamp_info]).to_csv(
    os.path.join(results_dir, 'analyzed_files.csv'),
    index=False
)

# Save feature data
np.save(os.path.join(results_dir, 'csi_windows.npy'), results['csi_windows'])
np.save(os.path.join(results_dir, 'audio_windows.npy'), results['audio_windows'])

# Save correlation matrix
cross_corr.to_csv(os.path.join(results_dir, 'cross_correlations.csv'))

# Save feature importance results
csi_importance.to_csv(os.path.join(results_dir, 'csi_feature_importance.csv'), index=False)
audio_importance.to_csv(os.path.join(results_dir, 'audio_feature_importance.csv'), index=False)

print("Analysis results saved to:", results_dir)
print("\nSaved files:")
for file in os.listdir(results_dir):
    print(f"- {file}")
